# MLOps — Pipelines & CI/CD — AI Lab Instructor Notebook

*MLOps & Systems · Expert*



8 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import json
from pathlib import Path
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup

# MLOps — Pipelines & CI/CD — Student Lab

Offline lab: build a tiny reproducible ML pipeline and add CI-style checks.

## 0 — Config + pipeline entrypoint

In [ ]:
import json
from pathlib import Path
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # MLOps — Pipelines & CI/CD — Instructor Lab

Offline lab: build a tiny reproducible ML pipeline and add CI-style checks.

## 0 — Config + pipeline entrypoint

## Task 2: 0 — Config + pipeline entrypoint

## 0 — Config + pipeline entrypoint

In [ ]:
CONFIG = {
    "seed": 0,
    "n": 3000,
    "d": 12,
    "train_frac": 0.8,
    "lr": 0.2,
    "steps": 400,
    "l2": 1e-2,
    "acc_gate": 0.80,
    "artifacts_dir": "artifacts",
}

def run_pipeline(cfg: dict):
    # Will be completed after the helper functions are implemented below.
    raise NotImplementedError("Implement helper steps first, then wire them together in Task 5.")


## Task 3: 1 — Data generation + schema checks

## 1 — Data generation + schema checks

In [ ]:
def make_synthetic(n: int, d: int, seed: int):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, d))
    w_true = rng.standard_normal((d,))
    b_true = 0.2
    logits = X @ w_true + b_true
    p = 1.0 / (1.0 + np.exp(-logits))
    y = (p > 0.5).astype(np.int64)
    return X.astype(np.float64), y

def train_test_split(X, y, train_frac: float, seed: int):
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = rng.permutation(n)
    nt = int(train_frac * n)
    tr, te = idx[:nt], idx[nt:]
    return X[tr], y[tr], X[te], y[te]

def schema_checks(X, y):
    check("X_2d", X.ndim == 2)
    check("y_1d", y.ndim == 1)
    check("n_match", X.shape[0] == y.shape[0])
    check("no_nan", np.isfinite(X).all() and np.isfinite(y).all())
    check("binary_y", set(np.unique(y).tolist()) <= {0,1})


## Task 4: 2 — Logistic regression (NumPy)

## 2 — Logistic regression (NumPy)

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def log_loss(y, p, eps: float = 1e-12):
    p = np.clip(p, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p) + (1-y) * np.log(1-p)))

def fit_logreg(X, y, lr: float, steps: int, l2: float, seed: int):
    rng = np.random.default_rng(seed)
    n, d = X.shape
    w = 0.01 * rng.standard_normal((d,))
    b = 0.0
    for _ in range(steps):
        z = X @ w + b
        p = sigmoid(z)
        # gradients
        gw = (X.T @ (p - y)) / n + l2 * w
        gb = float(np.mean(p - y))
        w -= lr * gw
        b -= lr * gb
    return {"w": w, "b": b}

def predict_proba(model, X):
    return sigmoid(X @ model["w"] + model["b"])

def predict(model, X):
    return (predict_proba(model, X) >= 0.5).astype(np.int64)


## Task 5: 3 — Evaluation + quality gate

## 3 — Evaluation + quality gate

In [ ]:
def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

def eval_model(model, X, y):
    p = predict_proba(model, X)
    yhat = (p >= 0.5).astype(np.int64)
    return {
        "acc": accuracy(y, yhat),
        "log_loss": log_loss(y, p),
    }

def quality_gate(metrics: dict, acc_gate: float):
    check("acc_gate", metrics["acc"] >= acc_gate)


## Task 6: 4 — Artifacts (model + metrics)

## 4 — Artifacts (model + metrics)

In [ ]:
def save_artifacts(cfg: dict, model: dict, metrics: dict):
    outdir = Path(cfg["artifacts_dir"])
    outdir.mkdir(parents=True, exist_ok=True)
    # Save weights (numpy) + metadata (json)
    np.save(outdir / "weights.npy", model["w"])
    (outdir / "bias.txt").write_text(str(float(model["b"])), encoding="utf-8")
    (outdir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    (outdir / "config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    return str(outdir)


## Task 7: 5 — Implement `run_pipeline` + CI-style checks

## 5 — Implement `run_pipeline` + CI-style checks

In [ ]:
def run_pipeline(cfg: dict):
    X, y = make_synthetic(cfg["n"], cfg["d"], seed=cfg["seed"])
    schema_checks(X, y)
    Xtr, ytr, Xte, yte = train_test_split(X, y, cfg["train_frac"], seed=cfg["seed"])
    model = fit_logreg(Xtr, ytr, lr=cfg["lr"], steps=cfg["steps"], l2=cfg["l2"], seed=cfg["seed"])
    metrics = eval_model(model, Xte, yte)
    quality_gate(metrics, acc_gate=cfg["acc_gate"])
    path = save_artifacts(cfg, model, metrics)
    metrics = {**metrics, "artifacts_dir": path}
    return model, metrics

model, metrics = run_pipeline(CONFIG)
print(metrics)
check('artifacts_dir_exists', Path(metrics['artifacts_dir']).exists())


In [ ]:
# Checks
check('artifacts_dir_exists', Path(metrics['artifacts_dir']).exists())

## Task 8: 6 — CI simulation: determinism + regression gate

## 6 — CI simulation: determinism + regression gate

In [ ]:
def almost_equal(a, b, tol=1e-12):
    return abs(float(a) - float(b)) <= tol

# Determinism check: same config should yield same metrics
_, m1 = run_pipeline(CONFIG)
_, m2 = run_pipeline(CONFIG)
check('deterministic_acc', almost_equal(m1['acc'], m2['acc']))
check('deterministic_loss', almost_equal(m1['log_loss'], m2['log_loss']))

# Regression simulation: tighten gate until it fails
bad = dict(CONFIG)
bad["acc_gate"] = 0.99
try:
    run_pipeline(bad)
    raise AssertionError("expected gate to fail")
except AssertionError as e:
    check('regression_gate_blocks_bad_run', 'acc_gate' in str(e))
    print('Expected regression gate block triggered on acc_gate')


In [ ]:
# Checks
check('deterministic_acc', almost_equal(m1['acc'], m2['acc']))
check('deterministic_loss', almost_equal(m1['log_loss'], m2['log_loss']))